# 插件开发实战

## OpenClaw 插件体系

OpenClaw 的插件是业务能力的最小单元。每个插件遵循统一的**插件协议（Plugin Protocol）**，通过基类继承实现标准化接口。

### 插件接口协议

```
BasePlugin (抽象基类)
  ├── name: str              # 插件唯一标识
  ├── description: str       # 插件功能描述（供 LLM 理解）
  ├── version: str           # 版本号
  ├── execute(params) → dict # 核心执行方法
  ├── validate(params) → bool# 参数校验
  └── health_check() → bool  # 健康检查
```

### 设计要点

| 原则 | 说明 | B站场景举例 |
|------|------|-------------|
| **单一职责** | 每个插件只做一件事 | ad_query 只查广告数据 |
| **自描述** | description 供 LLM 选择工具 | "查询广告投放的ROI、CPC等数据" |
| **参数校验** | execute 前先 validate | 校验广告主ID是否合法 |
| **幂等性** | 重复调用结果一致 | 查询类插件天然幂等 |

In [ ]:
# 实现一个完整的 ad_query 广告查询插件

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Any

# ---- 插件基类（OpenClaw Plugin Protocol）----
class BasePlugin(ABC):
    """OpenClaw 插件抽象基类，所有插件必须继承此类"""
    
    @property
    @abstractmethod
    def name(self) -> str:
        """插件唯一标识"""
        ...
    
    @property
    @abstractmethod
    def description(self) -> str:
        """功能描述，供 LLM 理解该插件能力"""
        ...
    
    @property
    def version(self) -> str:
        return "1.0.0"
    
    def validate(self, params: dict) -> bool:
        """参数校验，默认通过"""
        return True
    
    @abstractmethod
    def execute(self, params: dict) -> dict:
        """核心执行方法"""
        ...
    
    def health_check(self) -> bool:
        """健康检查"""
        return True
    
    def to_tool_schema(self) -> dict:
        """导出为 LLM Function Calling 格式"""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self._get_param_schema()
            }
        }
    
    def _get_param_schema(self) -> dict:
        return {"type": "object", "properties": {}, "required": []}


# ---- 广告查询插件实现 ----
class AdQueryPlugin(BasePlugin):
    """B站广告数据查询插件"""
    
    # 模拟数据库
    _MOCK_DATA = {
        "ad_001": {"campaign": "春季护肤品推广", "impressions": 50000, "clicks": 1200,
                   "cost": 8500.0, "conversions": 180, "roi": 2.35},
        "ad_002": {"campaign": "游戏联运CPS", "impressions": 120000, "clicks": 3600,
                   "cost": 25000.0, "conversions": 450, "roi": 3.10},
    }
    
    @property
    def name(self) -> str:
        return "ad_query"
    
    @property
    def description(self) -> str:
        return "查询B站商业化广告投放数据，包括曝光量、点击量、花费、ROI等指标"
    
    def validate(self, params: dict) -> bool:
        if "ad_id" not in params:
            print("  [校验失败] 缺少 ad_id 参数")
            return False
        if params["ad_id"] not in self._MOCK_DATA:
            print(f"  [校验失败] ad_id '{params['ad_id']}' 不存在")
            return False
        return True
    
    def execute(self, params: dict) -> dict:
        if not self.validate(params):
            return {"error": "参数校验失败"}
        
        ad_id = params["ad_id"]
        data = self._MOCK_DATA[ad_id]
        
        # 计算衍生指标
        data["cpc"] = round(data["cost"] / data["clicks"], 2)
        data["ctr"] = round(data["clicks"] / data["impressions"] * 100, 2)
        data["cvr"] = round(data["conversions"] / data["clicks"] * 100, 2)
        
        return {"status": "success", "ad_id": ad_id, "data": data}
    
    def _get_param_schema(self) -> dict:
        return {
            "type": "object",
            "properties": {
                "ad_id": {"type": "string", "description": "广告计划ID"}
            },
            "required": ["ad_id"]
        }

# 测试插件
plugin = AdQueryPlugin()
print(f"插件名: {plugin.name}")
print(f"描述: {plugin.description}")
print(f"版本: {plugin.version}")
print(f"健康检查: {plugin.health_check()}")
print()

# 正常查询
result = plugin.execute({"ad_id": "ad_001"})
print("查询结果:")
for k, v in result["data"].items():
    print(f"  {k}: {v}")

print()
# 导出 Tool Schema（供 LLM Function Calling 使用）
import json
print("Tool Schema:")
print(json.dumps(plugin.to_tool_schema(), indent=2, ensure_ascii=False))

In [ ]:
# 插件注册与发现机制

class PluginRegistry:
    """插件注册中心：管理所有已注册插件的生命周期"""
    
    def __init__(self):
        self._plugins: dict[str, BasePlugin] = {}
    
    def register(self, plugin: BasePlugin) -> None:
        """注册插件"""
        if not isinstance(plugin, BasePlugin):
            raise TypeError(f"插件必须继承 BasePlugin，收到 {type(plugin)}")
        
        if not plugin.health_check():
            raise RuntimeError(f"插件 '{plugin.name}' 健康检查未通过")
        
        self._plugins[plugin.name] = plugin
        print(f"  [注册] {plugin.name} v{plugin.version} ✓")
    
    def discover(self, keyword: str = "") -> list[str]:
        """根据关键词发现插件（LLM 可用此能力选择工具）"""
        matches = []
        for name, plugin in self._plugins.items():
            if keyword == "" or keyword in plugin.description:
                matches.append(name)
        return matches
    
    def get(self, name: str) -> BasePlugin:
        """获取插件实例"""
        if name not in self._plugins:
            raise KeyError(f"插件 '{name}' 未注册")
        return self._plugins[name]
    
    def list_tools(self) -> list[dict]:
        """导出所有插件的 Tool Schema 列表"""
        return [p.to_tool_schema() for p in self._plugins.values()]
    
    def __len__(self) -> int:
        return len(self._plugins)


# 额外定义一个商单插件用于演示
class CommercialOrderPlugin(BasePlugin):
    @property
    def name(self) -> str:
        return "commercial_order"
    
    @property
    def description(self) -> str:
        return "查询UP主商业合作订单，包括花火平台商单数据"
    
    def execute(self, params: dict) -> dict:
        return {"status": "success", "orders": [{"brand": "某护肤品牌", "fee": 50000}]}


# 演示注册和发现流程
print("=" * 45)
print("插件注册与发现演示")
print("=" * 45)

registry = PluginRegistry()

# 注册插件
print("\n1. 注册插件:")
registry.register(AdQueryPlugin())
registry.register(CommercialOrderPlugin())
print(f"   已注册 {len(registry)} 个插件")

# 发现插件
print("\n2. 插件发现:")
print(f"   关键词'广告': {registry.discover('广告')}")
print(f"   关键词'UP主': {registry.discover('UP主')}")
print(f"   全部插件:    {registry.discover()}")

# 通过注册中心调用
print("\n3. 通过注册中心调用插件:")
plugin = registry.get("ad_query")
result = plugin.execute({"ad_id": "ad_002"})
print(f"   {result['data']['campaign']}: ROI={result['data']['roi']}")

## 插件设计模式

### 1. 模板方法模式（Template Method）
BasePlugin 定义了 `validate → execute → health_check` 的执行骨架，子类只需实现具体逻辑。

### 2. 策略模式（Strategy）
不同插件实现相同接口，Runner 根据 Agent 的决策动态选择调用哪个插件。

### 3. 注册表模式（Registry）
PluginRegistry 管理所有插件的注册与发现，支持：
- **按名称精确查找**：`registry.get("ad_query")`
- **按描述模糊发现**：`registry.discover("广告")` — 供 LLM 工具选择
- **批量导出 Schema**：`registry.list_tools()` — 传给 LLM 的 tools 参数

### B站商业化插件矩阵

```
├── ad_query          # 广告数据查询
├── ad_create          # 广告计划创建
├── commercial_order   # 花火商单查询
├── content_recommend  # 内容推荐
├── live_commerce      # 直播带货数据
├── user_profile       # 用户画像查询
└── report_generator   # 报表生成
```

## 面试速记

### Q1: OpenClaw 的插件协议包含哪些核心方法？

> `name`（唯一标识）、`description`（供 LLM 理解的功能描述）、`validate`（参数校验）、`execute`（核心执行）、`health_check`（健康检查）。其中 description 至关重要 -- 它直接影响 LLM 能否正确选择工具。

### Q2: 插件的 description 为什么重要？

> description 会被转换为 Function Calling 的 tool schema 传给 LLM。LLM 根据 description 判断该调用哪个插件。描述不清晰会导致工具选择错误，这是 Agent 系统中最常见的失败原因之一。

### Q3: 如何保证插件的可靠性？

> 四层保障：（1）`validate` 前置校验非法参数；（2）`health_check` 注册时验证插件可用性；（3）Runner 层的超时/重试/熔断；（4）插件设计遵循幂等性原则，重复调用结果一致。

### 记忆口诀
```
插件五要素：名称 + 描述 + 校验 + 执行 + 健康检查
description 写不好 → LLM 选错工具 → Agent 整体失败
```